モデルを学習させ、PTH形式で保存する

In [1]:
from accelerate import Accelerator
from torch.optim import AdamW
from tqdm.notebook import tqdm
from bitsandbytes import optim as bnb_optim
import torch
import torch.nn as nn

def train(model, dataloader, vocab_size, epochs=3, lr=3e-4, weight_decay=0.0, early_stop_loss=2.0):
    accelerator = Accelerator()
    device = accelerator.device

    model.to(device)
    optimizer = bnb_optim.AdamW8bit(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    batch_size = dataloader.batch_size
    model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)
    model.train()
    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f"Epoch: {epoch+1}", disable=not accelerator.is_local_main_process)
        for batch in pbar:
            input_ids = batch[:, :-1]
            labels = batch[:, 1:]

            outputs = model(input_ids)

            loss = criterion(outputs.reshape(-1, vocab_size), labels.reshape(-1))


            optimizer.zero_grad()
            accelerator.backward(loss)
            optimizer.step()

            pbar.set_postfix(loss=loss.item())

            if pbar.n*batch_size % 65536 == 0:
                torch.save(model.state_dict(), f"../data/{model.__class__.__module__}_{path.max_seq_len}.pth")
                if loss.item() < early_stop_loss:
                    print(f"Training stopped early at epoch {epoch+1}, batch {pbar.n*batch_size} due to train_loss < {early_stop_loss}")
                    return  # 訓練を中止

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import path as path
from dataset import JPNDataset
import fftModel

dataset = torch.load(f"../data/dataset_{path.max_seq_len}.pt", weights_only=False)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

model = fftModel.Decoder(vocab_size=22287, d_model=128, n_heads=4, n_layers=2, max_seq_len=path.max_seq_len, dropout=0.0)
if os.path.isfile(f"../data/{model.__class__.__module__}_{path.max_seq_len}.pth"): model.load_state_dict(torch.load(f"../data/{model.__class__.__module__}_{path.max_seq_len}.pth"))
train(model, dataloader, vocab_size=22287, epochs=3, lr=3e-4, weight_decay=0.01)

Epoch: 1:   0%|          | 0/48371 [00:00<?, ?it/s]

KeyboardInterrupt: 

推論

In [ ]:
import torch
import json
import path as path
from charTokenizer import CharTokenizer
import fftModel

model = fftModel.Decoder(vocab_size=22287, d_model=64, n_heads=4, n_layers=2, max_seq_len=path.max_seq_len, dropout=0.0)
model.load_state_dict(torch.load(f"../data/{model.__class__.__module__}_{path.max_seq_len}.pth"))
model.eval()

tokenizer = CharTokenizer()
with open(path.charTokenizer, 'r', encoding='utf-8') as f:
    tokenizer.vocab = json.load(f)
inputs = tokenizer(
    "荒れた内を避ける為か中間付近までは馬場の中央付近を走行。道中ペースを緩め脚をためると、そのまま直線も先頭でゴールした。逃げての上がりは32.9で上がり最速タイ。 武豊騎手は「ポンと出て、無理に引っ張ることもなく、マイペースで行けた。ラストでひと伸びして能力の高さを感じた」とコメント。ききょうステークス以来実に1年ぶりの勝利を飾り、素質の高さを見せた。続く逆瀬川ステークスでは前走の走りを評価されてか、古馬と同じ55kgの斤量を課された。チャンピオンズカップの裏開催であった為、武豊騎手から吉田隼人騎手に乗り替わり、朝日杯FS以来のコンビ結成となった。スムーズにゲートを出ると、そのまま内3番手を追走。最後直線は力強く抜け出して、2連勝でのOP入りを決めた。吉田隼人騎手は「約1年ぶりに乗せていただきましたが成長しています。出たなりでいい位置をキープできました。抜け出してから、左にもたれる癖はあるが、上がり勝負にも対応してくれました。競馬に幅が広がったし、これからが楽しみです」と振り返った。2023年（4歳）.明け4歳の始動戦に選ばれたのは東京芝2000mのリステッド戦である[白富士ステークスとされた。当日は前走ローズステークス2着と好走したサリエラに次ぐ2番人気に評価された。レースはドーブネが最内枠から好スタートでハナを奪い、武豊のエスコートで1000m59.9秒という絶妙な時計で逃げを打つ。直線に入り粘りの逃げで後続を離すかに思えたが、残り200mあたりから失速。最後は後方から末脚を伸ばしてきたサリエラに交わされた。それでも内を通って伸びてきていたヤマニンサルバムには抜かせず2着を確保した。レース後武豊は「楽なペースだったけどね…。1ハロンぐらい少し距離が長いのかな」とコメントを残し、2000ｍはドーブネにとって距離が長い可能性が示唆された。"
    , max_seq_len=path.max_seq_len, return_tensors="pt")
device = next(model.parameters()).device
inputs["input_ids"] = inputs["input_ids"].to(device)
with torch.no_grad():
    outputs = model(inputs["input_ids"])
    predictions = torch.argmax(outputs, dim=-1)
output_text = tokenizer.decode(predictions[0])
print(output_text)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import model
import fft2Model

def check_model_learns(model, vocab_size, seq_len=16, batch_size=4, epochs=1000, lr=3e-4):
    model.train()
    device = next(model.parameters()).device

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # ランダムな自然数列を生成 (0 ~ vocab_size-1)
        x = torch.randint(low=0, high=vocab_size, size=(batch_size, seq_len), device=device)
        # 目標は次トークン予測なのでシフトしたもの
        y = torch.roll(x, shifts=-1, dims=1)

        optimizer.zero_grad()
        logits = model(x)  # [B, T, vocab_size]

        # CrossEntropyLoss用に形状を変換
        logits_flat = logits.view(-1, vocab_size)  # [B*T, vocab_size]
        y_flat = y.view(-1)  # [B*T]

        loss = criterion(logits_flat, y_flat)
        loss.backward()
        optimizer.step()

        if (epoch+1)%10==0: print(f"Epoch {epoch+1}: loss={loss.item():.4f}")

# モデル定義（例）
vocab_size = 50
d_model = 256
n_heads = 4
n_layers = 2
max_seq_len = 256

model = fft2Model.Decoder(vocab_size, d_model, n_heads, n_layers, max_seq_len).to('cuda' if torch.cuda.is_available() else 'cpu')
check_model_learns(model, vocab_size=vocab_size, seq_len=max_seq_len)


Epoch 10: loss=3.9218
Epoch 20: loss=3.9253
Epoch 30: loss=3.9339
Epoch 40: loss=3.9182
Epoch 50: loss=3.9180
Epoch 60: loss=3.9226
Epoch 70: loss=3.9192
Epoch 80: loss=3.9162
Epoch 90: loss=3.9196
Epoch 100: loss=3.9165
Epoch 110: loss=3.9181
Epoch 120: loss=3.9195
Epoch 130: loss=3.9137
Epoch 140: loss=3.9194
Epoch 150: loss=3.9158
Epoch 160: loss=3.9190
Epoch 170: loss=3.9137
Epoch 180: loss=3.9161
Epoch 190: loss=3.9140
Epoch 200: loss=3.9132
Epoch 210: loss=3.9084
Epoch 220: loss=3.9068
Epoch 230: loss=3.9041
Epoch 240: loss=3.8358
Epoch 250: loss=3.5898
Epoch 260: loss=3.1406
Epoch 270: loss=2.6569
Epoch 280: loss=2.1920
Epoch 290: loss=1.9711
Epoch 300: loss=1.5547
Epoch 310: loss=1.3229
Epoch 320: loss=1.1658
Epoch 330: loss=1.0090
Epoch 340: loss=0.8253
Epoch 350: loss=0.6556
Epoch 360: loss=0.5632
Epoch 370: loss=0.4871
Epoch 380: loss=0.4417
Epoch 390: loss=0.3502
Epoch 400: loss=0.3416
Epoch 410: loss=0.3484
Epoch 420: loss=0.2452
Epoch 430: loss=0.2082
Epoch 440: loss=0.20